# Notebook 06: Parameter Phase Diagram for Constraint Fragmentation

This notebook turns the repo into a **parameterized regime map**.

Notebook 05 showed that fragmented current-sheet structure is robust across randomized runs.

Notebook 06 asks:

- where are the smooth, bounded-fragmented, and strongly driven regimes?
- how do diffusion, activity strength, and damping organize the final-state geometry?

We sweep three control parameters:

- \(\nu\): diffusion / smoothing
- \(\beta\): activity strength near the 45° gate
- \(\gamma\): damping of the transverse field

For each parameter point, we run a small randomized ensemble and average the final-state observables.

Core gate:

\[
\cos\theta \ge \frac{1}{\sqrt{1^2 + 1^2}}
\]

This notebook is a **fragmentation regime map**, not a thermodynamic phase diagram.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import deque

THRESHOLD = 1 / np.sqrt(2)

nx, ny = 320, 240
x = np.linspace(-6.0, 6.0, nx)
y = np.linspace(-3.0, 3.0, ny)
X, Y = np.meshgrid(x, y)

dx = x[1] - x[0]
dy = y[1] - y[0]

print("45° threshold =", THRESHOLD)
print("grid shape =", X.shape)

## 1. Shared helpers

These reuse the same structure as Notebooks 04–05, but the notebook now sweeps parameter grids.

In [ ]:
def normalize_field(Bx, By):
    mag = np.sqrt(Bx**2 + By**2)
    mag = np.where(mag == 0, 1.0, mag)
    return Bx / mag, By / mag

def multimode_sheet_perturbation(X, Y, amps, ks, phases, sigma=0.38):
    out = np.zeros_like(X)
    envelope = np.exp(-(Y**2) / (sigma**2))
    for a, k, p in zip(amps, ks, phases):
        out += a * np.sin(k * X + p)
    return envelope * out

def harris_target(X, Y, delta=0.35):
    return np.tanh(Y / delta)

def initial_field(X, Y, delta=0.35, base_eps=0.06, frag_amp=0.28, sigma=0.38, rng=None):
    if rng is None:
        rng = np.random.default_rng(0)

    Bx = np.tanh(Y / delta)

    By_background = base_eps * np.sin(1.2 * X) * np.exp(-(Y**2) / (1.0**2))

    ks = [1.0, 2.0, 3.6, 5.0]
    phases = rng.uniform(0, 2 * np.pi, size=len(ks))
    amps = frag_amp * np.array([1.0, 0.75, 0.55, 0.35])
    amps = amps * (1.0 + 0.15 * rng.standard_normal(len(ks)))

    By_fragment = multimode_sheet_perturbation(X, Y, amps, ks, phases, sigma=sigma)

    By = By_background + By_fragment
    return Bx, By

def random_seed_pattern(X, Y, base_amp=0.28, sigma=0.38, rng=None):
    if rng is None:
        rng = np.random.default_rng(0)
    ks = [1.0, 2.0, 3.6, 5.0]
    phases = rng.uniform(0, 2 * np.pi, size=len(ks))
    amps = base_amp * np.array([1.0, 0.75, 0.55, 0.35])
    amps = amps * (1.0 + 0.15 * rng.standard_normal(len(ks)))
    return multimode_sheet_perturbation(X, Y, amps, ks, phases, sigma=sigma)

def laplacian(F, dx, dy):
    return (
        (np.roll(F, 1, axis=0) - 2 * F + np.roll(F, -1, axis=0)) / dy**2 +
        (np.roll(F, 1, axis=1) - 2 * F + np.roll(F, -1, axis=1)) / dx**2
    )

def cross_sheet_cosine(Bx_hat, By_hat, shift=3):
    Bx_up = np.roll(Bx_hat, -shift, axis=0)
    By_up = np.roll(By_hat, -shift, axis=0)

    Bx_dn = np.roll(Bx_hat, shift, axis=0)
    By_dn = np.roll(By_hat, shift, axis=0)

    cos_map = Bx_up * Bx_dn + By_up * By_dn
    cos_map[:shift, :] = np.nan
    cos_map[-shift:, :] = np.nan
    return cos_map

def failed_mask(cos_map, threshold=THRESHOLD):
    return cos_map < threshold

def activity_map(cos_map, threshold=THRESHOLD):
    act = np.maximum(0.0, threshold - cos_map)
    return np.nan_to_num(act, nan=0.0)

def central_band_mask(mask, Y, half_width=0.8):
    return mask & (np.abs(Y) < half_width)

def connected_components(mask):
    visited = np.zeros_like(mask, dtype=bool)
    rows, cols = mask.shape
    count = 0

    for i in range(rows):
        for j in range(cols):
            if mask[i, j] and not visited[i, j]:
                count += 1
                q = deque([(i, j)])
                visited[i, j] = True

                while q:
                    r, c = q.popleft()
                    for rr, cc in [(r-1, c), (r+1, c), (r, c-1), (r, c+1)]:
                        if 0 <= rr < rows and 0 <= cc < cols:
                            if mask[rr, cc] and not visited[rr, cc]:
                                visited[rr, cc] = True
                                q.append((rr, cc))
    return count

def estimate_failed_width_vs_x(cos_map, y, threshold=THRESHOLD):
    widths = np.full(cos_map.shape[1], np.nan)
    for j in range(cos_map.shape[1]):
        col = cos_map[:, j]
        bad = np.isfinite(col) & (col < threshold)
        if np.any(bad):
            y_bad = y[bad]
            widths[j] = y_bad.max() - y_bad.min()
    return widths

def constraint_energy(cos_map, threshold=THRESHOLD):
    return float(np.nanmean(np.maximum(0.0, threshold - cos_map)))

def one_step(Bx, By, Bx_target, seed_pattern, rng, dt, dx, dy, nu, alpha, beta, gamma, noise_amp=0.0, noise_sigma=0.38):
    Bx_hat, By_hat = normalize_field(Bx, By)
    cos_map = cross_sheet_cosine(Bx_hat, By_hat, shift=3)
    act = activity_map(cos_map, threshold=THRESHOLD)

    sheet_noise = noise_amp * rng.standard_normal(X.shape) * np.exp(-(Y**2) / (noise_sigma**2))

    Bx_new = Bx + dt * (
        nu * laplacian(Bx, dx, dy)
        - alpha * (Bx - Bx_target)
    )

    By_new = By + dt * (
        nu * laplacian(By, dx, dy)
        + beta * act * seed_pattern
        - gamma * By
        + sheet_noise
    )

    Bx_new[0, :] = Bx_target[0, :]
    Bx_new[-1, :] = Bx_target[-1, :]
    By_new[0, :] = 0.0
    By_new[-1, :] = 0.0

    return Bx_new, By_new

## 2. Parameter grids

We sweep diffusion \(\nu\), activity strength \(\beta\), and damping \(\gamma\).

For each point we run a small ensemble and average the final metrics.

In [ ]:
delta = 0.35
frag_amp = 0.28
sigma_seed = 0.38

alpha = 0.25
dt = 0.0018
noise_amp = 0.004
noise_sigma = 0.38
n_steps = 150

nu_vals = [0.0006, 0.0010, 0.0014, 0.0018, 0.0024]
beta_vals = [0.4, 0.7, 1.0, 1.3, 1.6]
gamma_vals = [0.10, 0.18, 0.28]

n_runs_per_point = 5

print("nu values   =", nu_vals)
print("beta values =", beta_vals)
print("gamma values=", gamma_vals)
print("runs per point =", n_runs_per_point)

## 3. Single parameter-point runner

This returns the mean final observables for a mini-ensemble at one parameter point.

In [ ]:
def run_parameter_point(nu, beta, gamma, n_runs=5):
    final_pocket_counts = []
    final_width_stds = []
    final_constraint_energies = []
    final_failed_fractions = []
    critical_steps = []

    example_snapshot = None

    for run_id in range(n_runs):
        rng = np.random.default_rng(50000 + run_id)

        Bx_t, By_t = initial_field(
            X, Y, delta=delta, base_eps=0.06, frag_amp=frag_amp, sigma=sigma_seed, rng=rng
        )
        Bx_target = harris_target(X, Y, delta=delta)
        seed_pattern = random_seed_pattern(X, Y, base_amp=frag_amp, sigma=sigma_seed, rng=rng)

        failed_fraction_trace = []

        for step in range(n_steps + 1):
            Bx_hat_t, By_hat_t = normalize_field(Bx_t, By_t)
            cos_map_t = cross_sheet_cosine(Bx_hat_t, By_hat_t, shift=3)
            mask_t = failed_mask(cos_map_t)
            central_t = central_band_mask(mask_t, Y, half_width=0.8)

            failed_fraction_trace.append(float(np.nanmean(mask_t)))

            if step == n_steps:
                pocket_count = connected_components(np.nan_to_num(central_t, nan=False).astype(bool))
                widths_t = estimate_failed_width_vs_x(cos_map_t, y, threshold=THRESHOLD)

                final_pocket_counts.append(float(pocket_count))
                final_width_stds.append(float(np.nanstd(widths_t)) if np.any(np.isfinite(widths_t)) else np.nan)
                final_constraint_energies.append(constraint_energy(cos_map_t))
                final_failed_fractions.append(float(np.nanmean(mask_t)))

                if example_snapshot is None:
                    example_snapshot = {
                        "cos_map": cos_map_t.copy(),
                        "central_mask": central_t.copy(),
                    }

            if step < n_steps:
                Bx_t, By_t = one_step(
                    Bx_t, By_t, Bx_target, seed_pattern, rng,
                    dt=dt, dx=dx, dy=dy,
                    nu=nu, alpha=alpha, beta=beta, gamma=gamma,
                    noise_amp=noise_amp, noise_sigma=noise_sigma
                )

        critical_steps.append(int(np.argmax(np.gradient(failed_fraction_trace))))

    summary = {
        "mean_pocket_count": float(np.nanmean(final_pocket_counts)),
        "mean_width_std": float(np.nanmean(final_width_stds)),
        "mean_constraint_energy": float(np.nanmean(final_constraint_energies)),
        "mean_failed_fraction": float(np.nanmean(final_failed_fractions)),
        "mean_critical_step": float(np.nanmean(critical_steps)),
        "example_snapshot": example_snapshot,
    }
    return summary

## 4. Run the phase-map sweep

For each fixed \(\gamma\), we build 2D arrays over \((\beta, \nu)\).

In [ ]:
phase_maps = {}

for gamma in gamma_vals:
    pocket_map = np.zeros((len(beta_vals), len(nu_vals)))
    width_map = np.zeros((len(beta_vals), len(nu_vals)))
    energy_map = np.zeros((len(beta_vals), len(nu_vals)))
    failed_map = np.zeros((len(beta_vals), len(nu_vals)))
    crit_map = np.zeros((len(beta_vals), len(nu_vals)))

    example_snapshots = {}

    for i, beta in enumerate(beta_vals):
        for j, nu in enumerate(nu_vals):
            summary = run_parameter_point(nu=nu, beta=beta, gamma=gamma, n_runs=n_runs_per_point)

            pocket_map[i, j] = summary["mean_pocket_count"]
            width_map[i, j] = summary["mean_width_std"]
            energy_map[i, j] = summary["mean_constraint_energy"]
            failed_map[i, j] = summary["mean_failed_fraction"]
            crit_map[i, j] = summary["mean_critical_step"]

            if (beta, nu) in [
                (0.4, 0.0024),
                (1.0, 0.0014),
                (1.6, 0.0006),
            ]:
                example_snapshots[(beta, nu)] = summary["example_snapshot"]

    phase_maps[gamma] = {
        "pocket_map": pocket_map,
        "width_map": width_map,
        "energy_map": energy_map,
        "failed_map": failed_map,
        "crit_map": crit_map,
        "example_snapshots": example_snapshots,
    }

print("completed gamma values:", list(phase_maps.keys()))

## 5. Heatmaps: mean final pocket count

In [ ]:
for gamma in gamma_vals:
    plt.figure(figsize=(7, 5))
    plt.imshow(
        phase_maps[gamma]["pocket_map"],
        origin="lower",
        aspect="auto",
        extent=[min(nu_vals), max(nu_vals), min(beta_vals), max(beta_vals)]
    )
    plt.colorbar(label="mean final pocket count")
    plt.xlabel("nu (diffusion)")
    plt.ylabel("beta (activity strength)")
    plt.title(f"Pocket Count Regime Map (gamma={gamma})")
    plt.show()

## 6. Heatmaps: mean final constraint energy

In [ ]:
for gamma in gamma_vals:
    plt.figure(figsize=(7, 5))
    plt.imshow(
        phase_maps[gamma]["energy_map"],
        origin="lower",
        aspect="auto",
        extent=[min(nu_vals), max(nu_vals), min(beta_vals), max(beta_vals)]
    )
    plt.colorbar(label="mean final constraint energy")
    plt.xlabel("nu (diffusion)")
    plt.ylabel("beta (activity strength)")
    plt.title(f"Constraint-Energy Regime Map (gamma={gamma})")
    plt.show()

## 7. Heatmaps: mean final width variation

In [ ]:
for gamma in gamma_vals:
    plt.figure(figsize=(7, 5))
    plt.imshow(
        phase_maps[gamma]["width_map"],
        origin="lower",
        aspect="auto",
        extent=[min(nu_vals), max(nu_vals), min(beta_vals), max(beta_vals)]
    )
    plt.colorbar(label="mean final width std")
    plt.xlabel("nu (diffusion)")
    plt.ylabel("beta (activity strength)")
    plt.title(f"Width-Variation Regime Map (gamma={gamma})")
    plt.show()

## 8. Optional supporting heatmaps

In [ ]:
for gamma in gamma_vals:
    plt.figure(figsize=(7, 5))
    plt.imshow(
        phase_maps[gamma]["failed_map"],
        origin="lower",
        aspect="auto",
        extent=[min(nu_vals), max(nu_vals), min(beta_vals), max(beta_vals)]
    )
    plt.colorbar(label="mean final below-threshold fraction")
    plt.xlabel("nu (diffusion)")
    plt.ylabel("beta (activity strength)")
    plt.title(f"Below-Threshold Fraction Map (gamma={gamma})")
    plt.show()

In [ ]:
for gamma in gamma_vals:
    plt.figure(figsize=(7, 5))
    plt.imshow(
        phase_maps[gamma]["crit_map"],
        origin="lower",
        aspect="auto",
        extent=[min(nu_vals), max(nu_vals), min(beta_vals), max(beta_vals)]
    )
    plt.colorbar(label="mean critical-step proxy")
    plt.xlabel("nu (diffusion)")
    plt.ylabel("beta (activity strength)")
    plt.title(f"Critical-Step Proxy Map (gamma={gamma})")
    plt.show()

## 9. Representative snapshots

We show one example snapshot from three representative parameter points:

- smoother regime
- intermediate bounded-fragmentation regime
- stronger-drive regime

In [ ]:
representative_points = [
    (0.4, 0.0024, "smoother regime"),
    (1.0, 0.0014, "intermediate regime"),
    (1.6, 0.0006, "stronger-drive regime"),
]

gamma_ref = 0.18

for beta, nu, label in representative_points:
    snap = phase_maps[gamma_ref]["example_snapshots"][(beta, nu)]

    plt.figure(figsize=(9, 4.6))
    plt.imshow(
        snap["cos_map"],
        extent=[x.min(), x.max(), y.min(), y.max()],
        origin="lower",
        aspect="auto"
    )
    plt.colorbar(label="cross-sheet cosine")
    plt.contour(
        X, Y, snap["cos_map"],
        levels=[THRESHOLD],
        colors="white",
        linewidths=1.8,
        linestyles="--"
    )
    plt.xlabel("x")
    plt.ylabel("y")
    plt.title(f"{label}: cosine map (beta={beta}, nu={nu}, gamma={gamma_ref})")
    plt.show()

    plt.figure(figsize=(9, 4.6))
    plt.imshow(
        snap["central_mask"],
        extent=[x.min(), x.max(), y.min(), y.max()],
        origin="lower",
        aspect="auto"
    )
    plt.xlabel("x")
    plt.ylabel("y")
    plt.title(f"{label}: central failed region (beta={beta}, nu={nu}, gamma={gamma_ref})")
    plt.show()

## 10. Compact parameter summary table

This prints the averaged observables for every parameter point.

In [ ]:
for gamma in gamma_vals:
    print(f"\n--- gamma = {gamma} ---")
    for i, beta in enumerate(beta_vals):
        for j, nu in enumerate(nu_vals):
            print(
                f"beta={beta:.2f} | nu={nu:.4f} | "
                f"pockets={phase_maps[gamma]['pocket_map'][i, j]:.3f} | "
                f"width_std={phase_maps[gamma]['width_map'][i, j]:.4f} | "
                f"energy={phase_maps[gamma]['energy_map'][i, j]:.4f} | "
                f"failed_frac={phase_maps[gamma]['failed_map'][i, j]:.4f} | "
                f"crit_step={phase_maps[gamma]['crit_map'][i, j]:.2f}"
            )

## 11. Interpretation

Notebook 06 upgrades the repo from stochastic robustness to a real **regime map**.

What it supports:

> The 45°-gated current-sheet model exhibits parameter-dependent fragmentation regimes controlled by the balance between smoothing, activity strength, and damping.

What to look for in the heatmaps:

- high \(\nu\), low \(\beta\): smoother regime
- intermediate \(\nu\), intermediate \(\beta\): bounded-fragmentation regime
- low \(\nu\), high \(\beta\): stronger-drive regime

What this notebook still does **not** claim:

- a thermodynamic phase diagram
- exact solar-plasma parameter calibration
- a replacement for MHD or PIC simulation

Best interpretation:

> a parameterized fragmentation regime map for the geometric precursor model.